# Criba de motores TTS — Qwen3-TTS, un episodio entero en modo solo timbre

Prueba de producción: el episodio **035 (*Una mente en calma, un cuerpo en forma, una casa llena de amor*)** entero, unos 4–5 min, generado con Qwen3-TTS en modo solo timbre y con el formato de la fábrica: **un clip por intervención** (`001nivi.wav`, `002naval.wav`…), cada frase unida con 250 ms de silencio.

Como en modo solo timbre Nivi se parecía menos, sus bloques salen también en modo normal para comparar. Salida en `outputs/criba/episodio_035/`:

| Carpeta o fichero | Qué es |
|---|---|
| `qwen3tts_xvec/` | Los 4 clips en modo solo timbre |
| `qwen3tts_1.7b/` | Los 2 clips de Nivi en modo normal |
| `episodio_xvec.wav` | El episodio montado, todo en modo solo timbre |
| `episodio_nivi_normal.wav` | El episodio montado: Naval en solo timbre, Nivi en modo normal |

Escucha el episodio montado **de principio a fin**, como lo oiría tu audiencia. Fíjate en si la voz aguanta en el bloque largo de Naval (61 frases), si el ritmo se vuelve monótono y cuánto trabajo te daría en Audacity.

**GPU:** L4. **Tiempo estimado:** unos 20 min, incluida la instalación.

**Cómo se usa** (es igual en todos los cuadernos de la criba):

1. *Entorno de ejecución → Cambiar tipo de entorno* → **L4**.
2. Ejecuta la **celda 1** (instalación). Cuando termine, *Entorno de ejecución → Reiniciar sesión*.
3. Ejecuta las demás celdas **en orden**, de la 2 a la última. La celda 2 comprueba que estén las
   referencias y las frases antes de cargar ningún modelo: si dice `RESULTADO: REVISAR`, para ahí.
4. La última celda **desconecta la GPU sola** para que no siga gastando unidades.

Si la sesión se corta a medias, vuelve a empezar por la celda 1: lo que ya esté en Drive no se
genera otra vez.

Salida: `fine_tunned_naval/outputs/criba/<variante>/` en Drive, un `.wav` por item (`n01`, `v01`,
`b01`, `k01`…) más `registro.jsonl`, `parametros.json` y `entorno.json`.

In [ ]:
# 1. Instalación (después: Entorno de ejecución → Reiniciar sesión)
!pip install -q qwen-tts==0.1.1

In [ ]:
# 2. Preparar: código del repo, Drive, el episodio y las referencias
import os, sys, subprocess, json
from pathlib import Path
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path.insert(0, "/content/repo/criba")
import comun

comun.montar_drive()
comun.SALIDA = comun.DRIVE / "outputs" / "criba" / "episodio_035"
EP = json.loads(Path("/content/repo/episodios/2019/035-finally-wealthy/transcript.es.json").read_text(encoding="utf-8"))
# Un item por bloque de hablante, con el nombre de la fábrica: 001nivi, 002naval...
ITEMS = []
for n, b in enumerate(EP["contenido"], 1):
    voz = b["speaker"].lower()
    ITEMS.append({"id": f"{n:03d}{voz}", "voz": voz, "ref": voz, "frases": b["frases"]})
SILENCIO_MS = 250
print(EP["titulo"])
for it in ITEMS:
    print(f"  {it['id']}: {len(it['frases'])} frases, {sum(len(f) for f in it['frases'])} caracteres")
REFS = comun.comprobar(ITEMS)

In [ ]:
# 3. Cargar el modelo
import torch
from qwen_tts import Qwen3TTSModel
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
modelo = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0",
                                       dtype=DTYPE, attn_implementation="sdpa")
_prompts = {}

def hacer_sintetizar(solo_timbre):
    def sintetizar(texto, ref):
        clave = (ref["wav"], solo_timbre)
        if clave not in _prompts:
            _prompts[clave] = modelo.create_voice_clone_prompt(
                ref_audio=ref["wav"], ref_text=None if solo_timbre else ref["texto"], x_vector_only_mode=solo_timbre)
        wavs, sr = modelo.generate_voice_clone(text=texto, language="Spanish", voice_clone_prompt=_prompts[clave])
        return wavs[0], sr
    return sintetizar
print("OK,", DTYPE)

In [ ]:
# 4. Generar: todo en solo timbre, y los bloques de Nivi también en modo normal
PARAMS = {"modelo": "Qwen/Qwen3-TTS-12Hz-1.7B-Base", "language": "Spanish", "dtype": str(DTYPE)}
comun.generar("qwen3tts_xvec", hacer_sintetizar(True), ITEMS, REFS, SILENCIO_MS,
              params={**PARAMS, "x_vector_only_mode": True})
comun.generar("qwen3tts_1.7b", hacer_sintetizar(False), [it for it in ITEMS if it["voz"] == "nivi"], REFS,
              SILENCIO_MS, params={**PARAMS, "x_vector_only_mode": False})

In [ ]:
# 5. Montar el episodio para escucharlo de principio a fin (600 ms entre intervenciones)
import numpy as np, soundfile as sf

def montar(nombre, carpeta_de):
    trozos, sr = [], None
    for it in ITEMS:
        audio, sr_clip = sf.read(str(comun.SALIDA / carpeta_de(it) / f"{it['id']}.wav"), dtype="float32")
        sr = sr or sr_clip
        trozos += [audio, np.zeros(int(sr * 0.6), dtype=np.float32)]
    total = np.concatenate(trozos[:-1])
    sf.write(str(comun.SALIDA / nombre), total, sr, subtype="PCM_16")
    print(f"{nombre}: {len(total) / sr / 60:.1f} min")

montar("episodio_xvec.wav", lambda it: "qwen3tts_xvec")
montar("episodio_nivi_normal.wav", lambda it: "qwen3tts_1.7b" if it["voz"] == "nivi" else "qwen3tts_xvec")

In [ ]:
# Última celda: resumen y desconexión de la GPU
import json
for variante in ['qwen3tts_xvec', 'qwen3tts_1.7b']:
    reg = comun.SALIDA / variante / "registro.jsonl"
    if not reg.exists():
        print(f"{variante}: sin registro")
        continue
    filas = [json.loads(l) for l in reg.read_text(encoding="utf-8").splitlines()]
    ok = sum(f["ok"] for f in filas)
    print(f"{variante}: {ok} items correctos, {len(filas) - ok} con error")
comun.desconectar()